In [0]:
import os
from pyspark.sql import functions as F

# Define paths inside your Unity Catalog Volume
staging_path = "/Volumes/ecommerce_lakehouse/bronze/raw_files/staging/"
base_vol = "/Volumes/ecommerce_lakehouse/bronze/raw_files/"

# List all files in the staging directory using dbutils
files = [f.name for f in dbutils.fs.ls(staging_path) if f.name.endswith(".csv")]

print(f"Found {len(files)} files in staging: {files}\n")

# Define which tables are transactional (fact) vs reference (dimension)
fact_tables = [
    "olist_orders_dataset.csv",
    "olist_order_items_dataset.csv",
    "olist_order_payments_dataset.csv",
    "olist_order_reviews_dataset.csv",
]

for file_name in files:
  file_path = os.path.join(staging_path, file_name)
  table_key = file_name.replace(".csv", "")

  print(f"Processing: {file_name}")

  # Read the CSV file into a PySpark DataFrame
  df = (
      spark.read.format("csv")
      .option("header", "true")
      .option("inferSchema", "true")
      .load(file_path)
  )

  if file_name in fact_tables:
    # Split fact tables into 3 incremental batches (70% / 20% / 10%)
    b1, b2, b3 = df.randomSplit([0.7, 0.2, 0.1], seed=42)

    # Write Batch 1
    b1.write.mode("overwrite").option("header", "true").csv(
        f"{base_vol}batch_1/{table_key}/"
    )

    # Write Batch 2
    b2.write.mode("overwrite").option("header", "true").csv(
        f"{base_vol}batch_2/{table_key}/"
    )

    # Write Batch 3 (Optional: Simulate small updates for orders)
    if file_name == "olist_orders_dataset.csv":
      b3 = b3.withColumn(
          "order_status",
          F.when(F.col("order_status") == "processing", "delivered").otherwise(
              F.col("order_status")
          ),
      )
    b3.write.mode("overwrite").option("header", "true").csv(
        f"{base_vol}batch_3/{table_key}/"
    )

    print(
        f" -> Fact table split into batch_1, batch_2, and batch_3 successfully."
    )

  else:
    # Dimension/Reference tables go fully into batch_1
    df.write.mode("overwrite").option("header", "true").csv(
        f"{base_vol}batch_1/{table_key}/"
    )
    print(f" -> Reference table saved entirely to batch_1.")

print("\nAll 9 files successfully processed and batched!")

Found 9 files in staging: ['olist_customers_dataset.csv', 'olist_geolocation_dataset.csv', 'olist_order_items_dataset.csv', 'olist_order_payments_dataset.csv', 'olist_order_reviews_dataset.csv', 'olist_orders_dataset.csv', 'olist_products_dataset.csv', 'olist_sellers_dataset.csv', 'product_category_name_translation.csv']

Processing: olist_customers_dataset.csv
 -> Reference table saved entirely to batch_1.
Processing: olist_geolocation_dataset.csv
 -> Reference table saved entirely to batch_1.
Processing: olist_order_items_dataset.csv
 -> Fact table split into batch_1, batch_2, and batch_3 successfully.
Processing: olist_order_payments_dataset.csv
 -> Fact table split into batch_1, batch_2, and batch_3 successfully.
Processing: olist_order_reviews_dataset.csv
 -> Fact table split into batch_1, batch_2, and batch_3 successfully.
Processing: olist_orders_dataset.csv
 -> Fact table split into batch_1, batch_2, and batch_3 successfully.
Processing: olist_products_dataset.csv
 -> Reference

In [0]:
# Create a widget in Databricks so you can pass the batch name dynamically
dbutils.widgets.text("batch_name", "batch_1", "Select Batch Name")
